# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/varshitha922/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

###  Two paper findings + my methodology questions

**Finding 1: Target label and prediction.**

My research paper defines the target label as whether a content item received at least one Google Search Console click in the seven days after the scoring date. The label comes from the observed clicks in this future seven-day period. The client-holdout validation helps check whether the model can predict this outcome for clients not used during training. My methodology question is whether the seven-day outcome window is clearly separated from the feature period to avoid data leakage.

**Finding 2: Model and baseline comparison.**

My research paper reports a Random Forest ROC-AUC of 0.885923 and a transparent impressions baseline ROC-AUC of 0.915891 on the same test split. Both methods use the same target label. The client-holdout design makes this comparison useful for evaluating performance on unseen clients. However, the results do not show that the Random Forest outperformed the baseline. My methodology question is whether the results remain consistent across different time periods and client groups.

**Overall:** The validation supports evaluating predictions on held-out clients, but it does not prove that refreshing a page will cause better search performance.


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-09 Section 1 — Research Paper Findings Check

finding_1 = {
    "target": "At least one GSC click in the next 7 days",
    "label_source": "Observed Google Search Console clicks",
    "validation": "Client-holdout validation"
}

finding_2 = {
    "random_forest_roc_auc": 0.885923,
    "baseline_roc_auc": 0.915891,
    "training_clients": 41,
    "held_out_clients": 11
}

print("Finding 1: Target and label")
for key, value in finding_1.items():
    print(key, ":", value)

print("\nFinding 2: Model comparison")
for key, value in finding_2.items():
    print(key, ":", value)

print("\nMethodology checks:")
print("1. Is the future 7-day label window separated from the feature period?")
print("2. Are results consistent across different time periods and client groups?")

print("\nConclusion: Baseline ROC-AUC is higher than Random Forest ROC-AUC.")

Finding 1: Target and label
target : At least one GSC click in the next 7 days
label_source : Observed Google Search Console clicks
validation : Client-holdout validation

Finding 2: Model comparison
random_forest_roc_auc : 0.885923
baseline_roc_auc : 0.915891
training_clients : 41
held_out_clients : 11

Methodology checks:
1. Is the future 7-day label window separated from the feature period?
2. Are results consistent across different time periods and client groups?

Conclusion: Baseline ROC-AUC is higher than Random Forest ROC-AUC.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

 ### My model under an honest split

In Week 5, I used a client-based grouped split, so the same client does not appear in both training and testing data. The original Random Forest Precision@50 was 0.58, and the grouped split rerun also measured 0.58. There were zero shared clients between the training and testing sets. This confirms that the result was reproducible under the same grouped split. However, it is not a comparison between two different validation methods.


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

# Load dataset
data_url = "https://raw.githubusercontent.com/varshitha922/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_url)

# Grouped split by client
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, groups=df["client_id"])
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

# Create labels
y_train = train_df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
y_test = test_df["trend_direction"].astype(str).str.lower().eq("down").astype(int)

# Select numeric features
excluded_words = [
    "id", "trend", "declin", "label", "future",
    "flag", "action", "score", "rank", "target",
    "last_30d", "prev_30d"
]

feature_cols = [
    col for col in train_df.select_dtypes(include=np.number).columns
    if not any(word in col.lower() for word in excluded_words)
    and col != "impressions_90d"
]

# Train Random Forest
model = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    )
)

model.fit(train_df[feature_cols], y_train)

# Predict scores
model_scores = model.predict_proba(test_df[feature_cols])[:, 1]

# Precision@50
def precision_at_50(y_true, scores):
    top_indices = np.argsort(-np.asarray(scores), kind="mergesort")[:50]
    return np.asarray(y_true)[top_indices].mean()

after_score = precision_at_50(y_test, model_scores)

# Compare original and rerun results
comparison = pd.DataFrame({
    "Evaluation": ["Week-5 original", "Grouped split rerun"],
    "Precision@50": [0.58, after_score]
})

display(comparison)

print("Shared clients:", len(
    set(train_df["client_id"]) & set(test_df["client_id"])
))

,Evaluation,Precision@50
0,Week-5 original,0.58
1,Grouped split rerun,0.58


Shared clients: 0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage audit

I checked the final feature set for possible data leakage. I excluded IDs, target-related fields, future information, and fields related to the outcome. I also checked whether any feature names contain suspicious terms. This helps ensure that the model uses only appropriate information available for prediction. The audit is based on feature names and does not guarantee that every feature is completely free from leakage.


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-09 Section 3 — Leakage Audit

# Check the final feature set
print("Total features:", len(feature_cols))

# Look for suspicious feature names
suspicious_words = [
    "future",
    "next",
    "target",
    "label",
    "trend",
    "declin",
    "outcome",
    "last_30d",
    "prev_30d"
]

suspicious_features = [
    col for col in feature_cols
    if any(word in col.lower() for word in suspicious_words)
]

print("\nSuspicious features found:")
print(suspicious_features)

# Check whether excluded fields entered the model
excluded_fields = [
    "client_id",
    "content_id",
    "trend_direction",
    "impressions_90d"
]

included_excluded_fields = [
    col for col in feature_cols
    if col in excluded_fields
]

print("\nExcluded fields included in model:")
print(included_excluded_fields)

# Final audit result
if len(suspicious_features) == 0 and len(included_excluded_fields) == 0:
    print("\nLeakage name check: PASSED")
else:
    print("\nLeakage name check: REVIEW REQUIRED")

Total features: 22

Suspicious features found:
[]

Excluded fields included in model:
[]

Leakage name check: PASSED


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim rewrite

**Original claim:** The Random Forest model identifies the best pages for content refresh and improves content performance.

**Rewritten claim:** In my experiment, the Random Forest model measured a Precision@50 of 0.58 on the held-out clients. This is an observed result from my dataset and validation split. The model provides directional scores that can help content teams decide which pages to review first. However, it does not guarantee that refreshing these pages will improve their performance.


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-09 Section 4 — Claim Rewrite

original_claim = (
    "The Random Forest model identifies the best pages for content "
    "refresh and improves content performance."
)

rewritten_claim = (
    "In my experiment, the Random Forest model measured a Precision@50 "
    "of 0.58 on the held-out clients. This is an observed result from "
    "my dataset and validation split. The model provides directional "
    "scores that can help content teams decide which pages to review "
    "first. However, it does not guarantee that refreshing these pages "
    "will improve their performance."
)

print("Original claim:")
print(original_claim)

print("\nRewritten claim:")
print(rewritten_claim)

Original claim:
The Random Forest model identifies the best pages for content refresh and improves content performance.

Rewritten claim:
In my experiment, the Random Forest model measured a Precision@50 of 0.58 on the held-out clients. This is an observed result from my dataset and validation split. The model provides directional scores that can help content teams decide which pages to review first. However, it does not guarantee that refreshing these pages will improve their performance.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.